In [ ]:
!pip install langchain-google-genai

In [ ]:
import os
from typing import List, Dict

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from google.colab import userdata

In [ ]:
os.environ["GOOGLE_API_KEY"]=userdata.get('GEMINI_API_KEY')

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0, streaming=True) # Enables token by token

summarizer_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2, streaming=False)

In [ ]:
system_prompt="""You are a professional travel agent.
You ONLY answer travel-related questions: flights, hotels, destinations, trips, itineraries, visas, packing and travel tips.
If the user asks anything non-travel-related, you MUST respond with exactly:
'I can't help with it.'
Do not explain this rule to the user.
"""

In [ ]:
travel_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=system_prompt),
        MessagesPlaceholder(variable_name="chat_history_for_llm"),
    ]
)

In [ ]:
summary_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system",
         "You are a helpful assisstant that summarizes chat histories."
         ),
        ("human",
         "Summarize the following travel-related conversation briefly but clearly,"
         "preserving important user preferences and decisions:\n\n{history}")
    ]
)

In [ ]:
chat_history: List = []

user_turn_count=0

def history_to_text(history: List) -> str:
  lines = []
  for msg in history:
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    if isinstance(msg, AIMessage):
      role = "System"
    lines.append(f"{role}: {msg.content}")
  return "\n".join(lines)

def summarize_history(history: List) -> List:
  if not history:
    return history

  history_text = history_to_text(history)
  summary_prompt = summary_prompt_template.format_prompt(history=history_text)
  summary_response = summarizer_llm.invoke(summary_prompt.to_messages())

  summary_message = SystemMessage(
      content=f"Conversation summary:\n{summary_response.content}"
  )

  return [summary_message]